# RAG（检索增强生成）指南

本文档汇总了RAG系统的技术栈。

---

## 目录

1. [RAG基本流程概述](#1-RAG基本流程概述)
2. [文档分片 (Chunking)](#2-文档分片-Chunking)
3. [文本向量化 (Embedding)](#3-文本向量化-Embedding)
4. [向量数据库与索引](#4-向量数据库与索引)
5. [相似度检索与召回](#5-相似度检索与召回)
6. [重排序 (Reranking)](#6-重排序-Reranking)
7. [生成 (Generation)](#7-生成-Generation)
8. [企业级高级技术](#8-企业级高级技术)
9. [技术选型建议](#9-技术选型建议)

---

## 1. RAG基本流程概述

RAG (Retrieval-Augmented Generation) 即检索增强生成，是一种将外部知识检索与大语言模型生成相结合的技术。

### 标准RAG流程

```
用户查询 → 文档分片 → 文本Embedding → 向量索引 → 相似度检索 → 召回文档 → 重排序 → LLM生成回答
```

### 本项目实现的RAG流程

本项目实现了两个RAG系统版本，分别是RAG1和RAG2，它们都实现了完整的RAG流程：文档分片 → 文本Embedding → 向量存储 → 相似度检索 → LLM生成回答。

#### RAG1 vs RAG2 对比

**相同点：**
- 系统目标：都是基于检索增强生成的中文文档问答系统
- 核心流程：都包含文档分片、文本嵌入、向量存储、检索召回、生成回答
- 向量数据库：都使用 ChromaDB
- 应用场景：都用于中文文档的智能问答

**不同点：**

| 特性 | RAG1 | RAG2 |
|------|------|------|
| **嵌入模型** | sentence-transformers (shibing624/text2vec-base-chinese) 本地模型 | 阿里云千问 text-embedding-v3 API |
| **向量存储** | ChromaDB EphemeralClient（内存临时存储） | ChromaDB PersistentClient（持久化存储） |
| **重排序模块** | Cross-Encoder (cross-encoder/mmarco-mMiniLMv2-L12-H384-v1) | 无 |
| **大语言模型** | Google Gemini 2.5 Flash | 阿里云千问 qwen-plus |
| **API** | Google API | 阿里云 DashScope |
| **文档分片** | 按 "\n\n" 简单分割 | 按章节分割，保留章节标题 |
| **依赖方式** | 本地模型，需要较大存储 | 云端 API，无需本地模型 |

**优缺点对比：**

| 特性 | RAG1 | RAG2 |
|------|------|------|
| **优点** | 本地模型可离线使用；Cross-Encoder重排序提高精度；灵活性高可替换模型 | 云端API无需部署；持久化存储；部署简单依赖少 |
| **缺点** | 需要下载存储本地模型占用磁盘空间；Ephemeral数据不持久化 | 需要稳定网络；无重排序精度略低；API调用有成本 |
| **适用场景** | 对检索精度要求高、需要离线部署、已有GPU资源 | 追求简单部署、快速原型开发、对成本敏感 |

---

## 2. 文档分片 (Chunking)

文档分片是将长文档分割成较小的文本块的过程，便于后续的嵌入和检索。

### 2.1 常见分片方法

In [2]:
# 示例文本
text = """在这里，我们有一段超过200字的中文文本作为输入例子。这段文本是关于自然语言处理的简介。
自然语言处理（NLP）是计算机科学、人工智能和语言学的交叉领域，它旨在让计算机能够理解和处理人类语言。
在这一领域中，机器学习技术扮演着核心角色。通过使用各种算法，计算机可以解析、理解、甚至生成人类可以理解的语言。
这一技术已广泛应用于机器翻译、情感分析、自动摘要、实体识别等多个方面。随着深度学习技术的发展，自然语言处理的准确性和效率都得到了显著提升。
当前，一些高级的NLP系统已经能够完成复杂的语言理解任务，例如问答系统、语音识别和对话系统等。
自然语言处理的研究不仅有助于改善人机交互，而且对于提高机器的自主性和智能化水平也具有重要意义。"""

### 方法1: 按句子切分

In [3]:
import re

# 按句子结束标点切分
sentences = re.split(r'(。|？！|\…\…)', text)
chunks = [sentence + (punc if punc else '') for sentence, punc in zip(sentences[::2], sentences[1::2])]

print("按句子切分结果:")
for i, chunk in enumerate(chunks):
    print(f"块{i+1}: {chunk}")

按句子切分结果:
块1: 在这里，我们有一段超过200字的中文文本作为输入例子。
块2: 这段文本是关于自然语言处理的简介。
块3: 
自然语言处理（NLP）是计算机科学、人工智能和语言学的交叉领域，它旨在让计算机能够理解和处理人类语言。
块4: 
在这一领域中，机器学习技术扮演着核心角色。
块5: 通过使用各种算法，计算机可以解析、理解、甚至生成人类可以理解的语言。
块6: 
这一技术已广泛应用于机器翻译、情感分析、自动摘要、实体识别等多个方面。
块7: 随着深度学习技术的发展，自然语言处理的准确性和效率都得到了显著提升。
块8: 
当前，一些高级的NLP系统已经能够完成复杂的语言理解任务，例如问答系统、语音识别和对话系统等。
块9: 
自然语言处理的研究不仅有助于改善人机交互，而且对于提高机器的自主性和智能化水平也具有重要意义。


### 方法2: 按固定字符数切分

In [4]:
def split_by_fixed_char_count(text, count):
    return [text[i:i+count] for i in range(0, len(text), count)]

chunks = split_by_fixed_char_count(text, 50)

print("按固定字符数(50)切分结果:")
for i, chunk in enumerate(chunks):
    print(f"块{i+1}: {chunk}")

按固定字符数(50)切分结果:
块1: 在这里，我们有一段超过200字的中文文本作为输入例子。这段文本是关于自然语言处理的简介。
自然语言处
块2: 理（NLP）是计算机科学、人工智能和语言学的交叉领域，它旨在让计算机能够理解和处理人类语言。
在这一
块3: 领域中，机器学习技术扮演着核心角色。通过使用各种算法，计算机可以解析、理解、甚至生成人类可以理解的语
块4: 言。
这一技术已广泛应用于机器翻译、情感分析、自动摘要、实体识别等多个方面。随着深度学习技术的发展，
块5: 自然语言处理的准确性和效率都得到了显著提升。
当前，一些高级的NLP系统已经能够完成复杂的语言理解任
块6: 务，例如问答系统、语音识别和对话系统等。
自然语言处理的研究不仅有助于改善人机交互，而且对于提高机器
块7: 的自主性和智能化水平也具有重要意义。


### 方法3: 按固定句子数切分

In [5]:
def split_by_fixed_sentence_count(sentences, count):
    return [sentences[i:i+count] for i in range(0, len(sentences), count)]

chunks = split_by_fixed_sentence_count(chunks, 3)

print("按固定句子数(3)切分结果:")
for i, chunk in enumerate(chunks):
    print(f"块{i+1}: {chunk}\n")

按固定句子数(3)切分结果:
块1: ['在这里，我们有一段超过200字的中文文本作为输入例子。这段文本是关于自然语言处理的简介。\n自然语言处', '理（NLP）是计算机科学、人工智能和语言学的交叉领域，它旨在让计算机能够理解和处理人类语言。\n在这一', '领域中，机器学习技术扮演着核心角色。通过使用各种算法，计算机可以解析、理解、甚至生成人类可以理解的语']

块2: ['言。\n这一技术已广泛应用于机器翻译、情感分析、自动摘要、实体识别等多个方面。随着深度学习技术的发展，', '自然语言处理的准确性和效率都得到了显著提升。\n当前，一些高级的NLP系统已经能够完成复杂的语言理解任', '务，例如问答系统、语音识别和对话系统等。\n自然语言处理的研究不仅有助于改善人机交互，而且对于提高机器']

块3: ['的自主性和智能化水平也具有重要意义。']



### 方法4: 使用LangChain的RecursiveCharacterTextSplitter

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,  # 块之间重叠，避免信息丢失
    length_function=len,
)

chunks = splitter.split_text(text)

print("RecursiveCharacterTextSplitter切分结果:")
for i, chunk in enumerate(chunks):
    print(f"块{i+1}: {chunk}")

RecursiveCharacterTextSplitter切分结果:
块1: 在这里，我们有一段超过200字的中文文本作为输入例子。这段文本是关于自然语言处理的简介。
自然语言处理（NLP）是计算机科学、人工智能和语言学的交叉领域，它旨在让计算机能够理解和处理人类语言。
块2: 在这一领域中，机器学习技术扮演着核心角色。通过使用各种算法，计算机可以解析、理解、甚至生成人类可以理解的语言。
块3: 这一技术已广泛应用于机器翻译、情感分析、自动摘要、实体识别等多个方面。随着深度学习技术的发展，自然语言处理的准确性和效率都得到了显著提升。
块4: 当前，一些高级的NLP系统已经能够完成复杂的语言理解任务，例如问答系统、语音识别和对话系统等。
自然语言处理的研究不仅有助于改善人机交互，而且对于提高机器的自主性和智能化水平也具有重要意义。


### 方法5： 按章节+标题分割（保留上下文）

In [7]:
# 保留章节标题
def get_chunks_with_header(content: str) -> list:
    """
    按章节分割，保留每个chunk前面的标题
    """
    chunks = content.split("\n\n")
    result = []
    header = ""
    for c in chunks:
        if c.startswith("#"):
            header += f"{c}\n"
        else:
            result.append(f"{header}{c}")
            header = ""
    return result

# 示例
sample_doc = """# 第一章 简介
这是简介部分的内容。

# 第二章 核心技术
核心技术包括机器学习和深度学习。

自然语言处理是重要研究方向。"""

chunks = get_chunks_with_header(sample_doc)
for i, c in enumerate(chunks):
    print(f"块{i+1}:\n{c}")
    print("-" * 30)

块1:
# 第一章 简介
这是简介部分的内容。
# 第二章 核心技术
核心技术包括机器学习和深度学习。
自然语言处理是重要研究方向。
------------------------------


### 分片方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|---------|
| 按句子 | 保持语义完整 | 可能过长 | 短文档 |
| 固定字符 | 统一大小 | 破坏语义 | 简单场景 |
| 固定句子 | 平衡大小和语义 | 可能不连续 | 通用场景 |
| RecursiveCharacterTextSplitter | 智能递归分割 | 需要调参 | 生产环境 |
| 章节+标题 | 保留上下文 | 需要结构化文档 | 技术文档 |

### 分片参数建议
- **chunk_size**: 常见值 256-1024 字符
- **chunk_overlap**: 通常设为 chunk_size 的 10-20%，保证上下文连贯

---

## 3. 文本向量化 (Embedding)

将文本转换为向量表示，使语义相似的文本在向量空间中距离更近。

In [ ]:
# 示例：使用阿里云千问API获取Embedding
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ.get('api_key', 'your-api-key'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

def get_embedding(text, model="text-embedding-v3"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

emb = get_embedding("自然语言处理是人工智能的重要分支")
print(f"向量维度: {len(emb)}")
print(f"向量前10维: {emb[:10]}")

### 3.1 主流Embedding模型

| 模型 | 维度 | 说明 | 使用方式 |
|------|------|------|---------|
| text-embedding-v3 (阿里云) | 1024/1536 | 支持自定义维度 | API调用 |
| shibing624/text2vec-base-chinese | 768 | 中文本地模型 | sentence-transformers |
| bge-large-zh-v1.5 | 1024 | BGE中文旗舰模型 | API/本地 |
| m3e-base | 768 | 阿里开源中文模型 | 本地 |
| Gemini Embedding | 768+ | Google模型 | API调用 |

### 3.2 Embedding任务类型

不同的任务类型使用不同的task_type：
- **RETRIEVAL_DOCUMENT**: 文档嵌入，适合建库
- **RETRIEVAL_QUERY**: 查询嵌入，适合检索
- **SEMANTIC_SIMILARITY**: 语义相似度
- **CLASSIFICATION**: 分类
- **CLUSTERING**: 聚类

---

## 4. 向量数据库与索引

向量数据库用于存储文本块及其对应的嵌入向量，支持高效的相似度检索。

In [ ]:
# ChromaDB 示例
import chromadb

# 临时客户端（内存存储，重启后丢失）
ephemeral_client = chromadb.EphemeralClient()
ephemeral_collection = ephemeral_client.get_or_create_collection("test")

# 持久化客户端（数据保存到磁盘）
persistent_client = chromadb.PersistentClient(path="./chroma.db")
persistent_collection = persistent_client.get_or_create_collection("test")

# 添加数据
documents = ["文档1内容", "文档2内容", "文档3内容"]
embeddings = [[0.1]*768, [0.2]*768, [0.3]*768]
persistent_collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=["1", "2", "3"]
)

print("数据已添加到向量数据库")

### 4.1 主流向量数据库

| 数据库 | 类型 | 特点 | 适用场景 |
|--------|------|------|---------|
| **ChromaDB** | 开源 | 轻量易用，Python优先 | 原型/小规模 |
| **Milvus** | 开源 | 高性能，支持亿级向量 | 企业级大规模 |
| **Qdrant** | 开源 | Rust实现，高性能 | 生产环境 |
| **Pinecone** | SaaS | 托管服务，易扩展 | 云部署 |
| **Weaviate** | 开源 | GraphQL支持 | 复杂查询 |
| **Faiss** | 开源 | Facebook开源，内存级 | 研究/小规模 |
| **阿里云向量检索** | SaaS | 阿里云生态 | 国内企业 |
| **腾讯云向量数据库** | SaaS | 腾讯云生态 | 国内企业 |

### 4.2 索引算法

| 算法 | 特点 | 适用场景 |
|------|------|---------|
| **HNSW** | 高速搜索，内存密集 | 召回率优先 |
| **IVF** | 分组搜索，平衡速度精度 | 大规模数据 |
| **PQ** | 压缩存储，降低内存 | 存储受限 |
| **Flat** | 精确搜索 | 小规模数据 |

---

## 5. 相似度检索与召回

通过计算查询向量与文档向量的相似度，检索最相关的内容。

In [ ]:
import numpy as np

# 余弦相似度计算
def cosine_similarity(A, B):
    dot_product = np.dot(A, B)
    norm_A = np.linalg.norm(A)
    norm_B = np.linalg.norm(B)
    return dot_product / (norm_A * norm_B)

# 示例
emb1 = get_embedding("大模型的应用场景")
emb2 = get_embedding("人工智能技术应用")
emb3 = get_embedding("python编程")

print(f"相似语义余弦相似度: {cosine_similarity(emb1, emb2):.4f}")
print(f"不同语义余弦相似度: {cosine_similarity(emb1, emb3):.4f}")

### 5.1 相似度度量方式

| 方法 | 说明 | 适用场景 |
|------|------|---------|
| **余弦相似度** | 衡量方向相似性，最常用 | 文本语义匹配 |
| **欧氏距离** | 衡量绝对距离 | 特征空间 |
| **点积** | 考虑 magnitude | 排序场景 |
| **曼哈顿距离** | 城市网格距离 | 特定场景 |

### 5.2 ChromaDB检索示例

In [ ]:
# 从向量数据库检索
def retrieve_from_chroma(query, collection, top_k=5):
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0], results.get('distances', [[]])[0]

# 使用示例
# retrieved_docs, distances = retrieve_from_chroma("你的问题", persistent_collection)
# for doc, dist in zip(retrieved_docs, distances):
#     print(f"相似度: {1-dist:.4f}, 内容: {doc}")

### 5.3 召回策略

| 策略 | 说明 | 优点 |
|------|------|------|
| **相似度阈值** | 只召回超过阈值的文档 | 减少噪音 |
| **Top-K召回** | 召回前K个最相似文档 | 简单高效 |
| **多路召回** | 同时使用向量召回+关键词召回 | 全面覆盖 |
| **混合召回** | 结合BM25和向量检索 | 平衡精确性和泛化 |

---

## 6. 重排序 (Reranking)

对初步检索结果进行二次排序，提高相关性。

In [ ]:
# RAG1 使用的 Cross-Encoder 重排序
from sentence_transformers import CrossEncoder

def rerank(query, retrieved_chunks, top_k=3):
    """
    使用Cross-Encoder对检索结果进行重排序
    """
    cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)
    
    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)
    
    return [chunk for chunk, _ in scored_chunks][:top_k]

# 示例
# reranked = rerank("你的问题", retrieved_chunks)
# for i, chunk in enumerate(reranked):
#     print(f"排名{i+1}: {chunk}")

### 6.1 主流重排序模型

| 模型 | 说明 | 特点 |
|------|------|------|
| cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 | 微软MiniLM | 轻量快速 |
| cross-encoder/ms-marco-MiniLM-L-12-v2 | 微软 | 精度高 |
| BAAI/bge-reranker-base | BGE | 中文友好 |
| BAAI/bge-reranker-large | BGE | 精度最高 |
| Cohere rerank | Cohere | 商业API |

### 6.2 Bi-Encoder vs Cross-Encoder

| 特性 | Bi-Encoder | Cross-Encoder |
|------|------------|---------------|
| 速度 | 快（可预计算） | 慢（实时计算） |
| 精度 | 较低 | 较高 |
| 用途 | 初次检索 | 重排序 |
| 向量 | 可存储 | 无需存储 |

---

## 7. 生成 (Generation)

将检索到的内容作为上下文，调用大语言模型生成最终回答。

In [ ]:
# 构建Prompt模板
def build_prompt(query, chunks):
    """
    构建RAG问答提示词
    """
    prompt = f"""你是一位知识助手，请根据用户的问题和下列片段生成准确的回答。

用户问题: {query}

相关片段:
{"\n\n".join(chunks)}

请基于上述内容作答，不要编造信息。如果无法从片段中找到答案，请如实告知。"""
    return prompt

# 示例
query = "什么是自然语言处理？"
chunks = ["NLP是计算机科学、人工智能和语言学的交叉领域", "NLP旨在让计算机理解和处理人类语言"]
prompt = build_prompt(query, chunks)
print("生成的Prompt:")
print(prompt)

### 7.1 主流LLM对比

| 模型 | 提供商 | 特点 | 适用场景 |
|------|--------|------|---------|
| GPT-4o | OpenAI | 综合能力强 | 通用问答 |
| Claude 3.5 | Anthropic | 长文本理解 | 复杂推理 |
| Gemini 2.5 | Google | 多模态 | 综合应用 |
| qwen-plus | 阿里云 | 中文优化 | 国内企业 |
| DeepSeek-V3 | 深度求索 | 开源免费 | 成本敏感 |
| 文心一言 | 百度 | 中文生态 | 国内企业 |
| 通义千问 | 阿里云 | 中文优化 | 国内企业 |
| 智谱清言 | 智谱AI | 中文优化 | 国内企业 |

### 7.2 Prompt工程技巧

1. **明确指令**: 清晰说明角色和任务
2. **提供上下文**: 包含检索到的相关内容
3. **约束输出**: 指定回答格式或限制
4. **少样本提示**: 提供示例帮助模型理解
5. **思维链**: 让模型先思考再回答

---

## 8. 企业级高级技术

本节介绍在生产环境中常用的RAG高级技术。

### 8.1 高级分块策略

#### 8.1.1 Markdown分块

针对Markdown文档，保留标题层级结构。

In [ ]:
# Markdown分块示例
from langchain.text_splitter import MarkdownTextSplitter

markdown_text = """# 第一章
## 第一节
内容一
## 第二节
内容二
# 第二章
内容三"""

splitter = MarkdownTextSplitter(chunk_size=100, chunk_overlap=0)
chunks = splitter.split_text(markdown_text)
for i, c in enumerate(chunks):
    print(f"块{i+1}: {c}")

#### 8.1.2 代码分块

针对代码文档，保持代码结构完整。

In [ ]:
from langchain.text_splitter import Language

python_code = """def hello():
    print("Hello")
    return True

class MyClass:
    def __init__(self):
        pass"""

splitter = LanguageTextSplitter(language=Language.PYTHON, chunk_size=200, chunk_overlap=0)
chunks = splitter.split_text(python_code)
for i, c in enumerate(chunks):
    print(f"块{i+1}:\n{c}\n")

#### 8.1.3 父子文档分块

维护父子层级关系，先检索小chunk，再检索相关大chunk。

### 8.2 多路召回

同时使用多种检索方法，综合结果。

In [ ]:
# 多路召回示例框架
def multi_way_retrieval(query, top_k=5):
    """
    多路召回：向量检索 + 关键词检索
    """
    # 1. 向量检索
    vector_results = vector_search(query, top_k*2)
    
    # 2. 关键词检索 (BM25)
    keyword_results = bm25_search(query, top_k*2)
    
    # 3. 结果融合 (RRF - Reciprocal Rank Fusion)
    fused = rrf_fusion(vector_results, keyword_results)
    
    return fused[:top_k]

# RRF融合算法
def rrf_fusion(results_list, k=60):
    """倒数排序融合算法"""
    doc_scores = {}
    for results in results_list:
        for rank, doc in enumerate(results):
            doc_scores[doc] = doc_scores.get(doc, 0) + 1 / (k + rank + 1)
    
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in sorted_docs]

### 8.3 查询改写

将用户口语化查询转换为检索友好的查询。

In [ ]:
# 查询改写示例
def rewrite_query(query, llm):
    """
    将用户查询改写为更适合检索的形式
    """
    prompt = f"""请将以下用户查询改写为更适合信息检索的形式，保留核心语义，去除口语化表达。

原始查询: {query}

请直接输出改写后的查询，不要其他解释。"""
    return llm.generate(prompt)

# 示例
# 原始: "我想了解一下那个什么机器学习到底是个啥玩意儿"
# 改写后: "机器学习的定义和概念"

### 8.4 HyDE (Hypothetical Document Embeddings)

让LLM先生成一个假设性答案，再用这个答案去检索。

In [ ]:
# HyDE流程
def hyde_retrieval(query, llm, vector_db, top_k=5):
    """
    1. 让LLM生成一个假设性答案
    2. 用假设答案进行向量检索
    3. 返回检索结果
    """
    # Step 1: 生成假设答案
    hypothetical_doc = llm.generate(f"请生成一个回答以下问题的示例答案：{query}")
    
    # Step 2: 用假设答案检索
    results = vector_db.search(hypothetical_doc, top_k)
    
    return results

### 8.5 索引结构优化

#### 8.5.1 分层索引

创建两层索引：快速粗筛 + 精确精筛。

#### 8.5.2 知识图谱增强

将实体关系融入检索，提升准确性。

In [ ]:
# 知识图谱增强检索示例
def kg_enhanced_retrieval(query, vector_db, kg_db):
    """
    1. 从查询中提取实体
    2. 从知识图谱获取相关实体
    3. 将实体信息加入查询
    4. 向量检索
    """
    # entities = extract_entities(query)  # 提取实体
    # related_entities = kg_db.get_related(entities)  # 获取相关实体
    # enhanced_query = f"{query} {' '.join(related_entities)}"
    # return vector_db.search(enhanced_query)
    pass

### 8.6 Agent增强RAG

使用Agent动态决定检索策略和次数。

In [ ]:
# Agent RAG 示例框架
def agent_rag(query, llm, vector_db):
    """
    Agent自主决策的RAG流程：
    1. 分析问题类型
    2. 决定是否需要检索
    3. 制定检索策略
    4. 迭代检索直到满意
    5. 生成最终答案
    """
    # 是否需要检索
    need_retrieval = llm.decide(f"判断是否需要外部知识回答：{query}")
    
    if not need_retrieval:
        return llm.generate(query)
    
    # 检索并评估
    context = vector_db.search(query)
    
    # 检查是否需要再次检索
    quality = llm.evaluate(f"根据以下上下文判断能否回答问题：{query}\n{context}")
    
    if quality < 0.7:
        # 改写查询再次检索
        refined_query = llm.refine(query)
        context = vector_db.search(refined_query)
    
    # 生成最终答案
    return llm.generate_with_context(query, context)

### 8.7 完整RAG技术栈汇总表

| 环节 | 基础技术 | 高级技术 |
|------|---------|---------|
| **分块** | 固定字符/句子 | Markdown分块、代码分块、父子文档、语义分块 |
| **Embedding** | 单一路模型 | 多模型ensemble、HyDE |
| **索引** | 单一向量索引 | 分层索引、知识图谱索引、多路索引 |
| **检索** | Top-K相似度 | 多路召回、查询改写、Agent自适应检索 |
| **重排序** | Cross-Encoder | 多级重排、列表学习重排 |
| **生成** | 基础Prompt | 思维链提示、少样本提示、Prompt压缩 |

---

## 9. 技术选型建议

### 9.1 按场景选型

| 场景 | 推荐方案 |
|------|---------|
| **原型/学习** | ChromaDB + 本地Embedding + 开源LLM |
| **小型生产** | ChromaDB/Qdrant + API Embedding + GPT4/qwen |
| **中型企业** | Milvus/Qdrant + 多路召回 + Agent |
| **大型企业** | 定制RAG + 知识图谱 + Agent + 私有化部署 |

### 9.2 成本考量

| 方案 |  embedding成本 | LLM成本 | 部署成本 |
|------|---------------|---------|---------|
| 本地模型 | 低（一次性） | - | 高（GPU） |
| API调用 | 按量付费 | 按量付费 | 低 |
| 混合部署 | 中等 | 按量付费 | 中等 |

### 9.3 性能优化建议

1. **检索优化**: 调整chunk_size、添加重叠、使用混合检索
2. **速度优化**: 使用缓存、预计算、量化模型
3. **精度优化**: 添加重排序、使用Agent迭代
4. **成本优化**: 选用性价比模型、减少token用量

---

## 附录：快速参考代码模板

In [ ]:
"""
RAG快速参考模板
"""

# 1. 分块
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(document)

# 2. Embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-large-zh-v1.5')
embeddings = model.encode(chunks)

# 3. 向量存储
import chromadb
client = chromadb.PersistentClient(path='./db')
collection = client.get_or_create_collection('docs')
collection.add(documents=chunks, embeddings=embeddings, ids=[str(i) for i in range(len(chunks))])

# 4. 检索
query_embedding = model.encode([query])
results = collection.query(query_embeddings=query_embedding, n_results=5)

# 5. 生成
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[{'role': 'user', 'content': f'基于以下内容回答：{results["documents"][0]}\n问题：{query}'}]
)
print(response.choices[0].message.content)

---

## 参考资源

- LangChain: https://python.langchain.com/
- LangChain中文文档: https://www.langchain.com.cn/
- RAG论文: https://arxiv.org/abs/2005.11401
- ChromaDB: https://docs.trychroma.com/
- BGE模型: https://github.com/FlagAI-Open/FlagAI
- LlamaIndex: https://www.llamaindex.ai/